# 교안 01-1: 파일시스템 MCP 서버 붙이기

개념은 같은 폴더의 `교안_01_MCP_개념.ipynb` 를 먼저 읽으세요.

## 핵심 목표

남이 만들어 공개한 MCP 서버에 붙어, 우리가 한 줄도 만들지 않은 도구를 그대로 불러 쓴다.

## 학습 순서

1. 파일시스템 MCP 서버에 붙어 도구 목록 받기
2. 도구 명세(name·description·args) 읽기: 에이전트가 보는 정보와 같다
3. 도구 직접 호출과 반환값의 원래 모양
4. 반환값에서 사람이 읽을 부분(text) 꺼내기
5. 허용된 폴더 밖은 막힌다
6. 받은 도구를 **LangChain 에이전트에 붙이기**: 무엇을 부를지 모델이 정한다

## 쓰는 MCP 서버와 공식 문서

| 서버 | 실행 | 전송 | 공식 문서 |
|---|---|---|---|
| 파일시스템 `@modelcontextprotocol/server-filesystem` | `npx` | stdio | https://github.com/modelcontextprotocol/servers/tree/main/src/filesystem |

MCP 규격 자체: https://modelcontextprotocol.io/docs/getting-started/intro

## 준비물

- **Node.js**(`npx -v` 로 확인). 없으면 https://nodejs.org
- 인터넷. 첫 실행은 서버 패키지를 내려받느라 수십 초 걸립니다.
- **에이전트를 만드는 절부터 `OPENAI_API_KEY`** 가 필요합니다(일차 폴더의 `.env`). 그 앞은 키 없이도 됩니다.

## 노트북으로 실습할 때 달라지는 것

같은 폴더의 `.py` 판과 코드가 거의 같지만, 세 곳이 다릅니다.

| `.py` | 노트북 | 왜 |
|---|---|---|
| `asyncio.run(main())` | 셀에서 바로 `await` | 주피터 커널은 이미 이벤트 루프를 돌리고 있어 `asyncio.run` 을 쓰면 `RuntimeError` 가 난다 |
| `Path(__file__).parent` | `Path.cwd()` | 노트북에는 `__file__` 이 없다 |
| `async with client.session(...)` | `await client.get_tools(...)` | `async with` 블록은 셀이 끝나면 닫혀, 다음 셀에서 도구가 죽는다 |

---
## 준비

같은 폴더의 `utils.py` 에 있는 도우미를 그대로 가져다 씁니다(`.py` 판과 같은 함수입니다).
주피터는 **노트북이 있는 폴더**를 모듈 검색 경로에 넣어 주므로 `from utils import ...` 가 그대로 됩니다.

In [ ]:
import sys
from pathlib import Path

# 노트북에는 __file__ 이 없다. 주피터는 노트북이 있는 폴더를 작업 폴더로 잡아 주므로 그 위가 일차 폴더다.
DAY_DIR = Path.cwd().parent        # 일차 폴더(day21). 아래 경로들의 기준점
sys.path.append(str(DAY_DIR))   # 일차 폴더의 utils.py 를 쓴다

from pprint import pprint          # 리스트·딕셔너리를 줄 맞춰 보기 좋게 찍는다

from langchain.agents import create_agent
from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain_openai import ChatOpenAI

# block_text: MCP 반환값에서 사람이 읽을 문자열을 뽑는다
# print_trajectory: 에이전트가 어떤 도구를 어떤 인자로 불렀는지 기록을 찍는다(에이전트를 붙일 때 쓴다)
from utils import block_text, load_api_key, print_trajectory

DATA_DIR = DAY_DIR / "data"        # 실습에 쓰는 CSV·DB 가 있는 곳

load_api_key(DAY_DIR)              # 모델을 부르는 절이 있으므로 키를 맨 앞에서 확인한다

print("일차 폴더:", DAY_DIR)
print("데이터 폴더가 있나?:", DATA_DIR.exists())

---
## 서버 설정: 딕셔너리 한 개가 곧 "서버를 어떻게 띄울지"

MCP 클라이언트에는 **서버를 띄우는 방법**을 딕셔너리로 알려 줍니다. 아래 네 항목이 전부입니다.

| 키 | 값 | 뜻 |
|---|---|---|
| `command` | `"npx"` | 서버를 **띄울 실행기**. 이 서버는 Node 패키지라 `npx` 로 받아 실행한다. 파이썬 패키지면 `uvx`, 이미 설치된 프로그램이면 그 이름을 그대로 적는다 |
| `args[0]` | `"-y"` | `npx` 에게 **설치할지 묻지 말고 진행**하라는 뜻. 이게 없으면 첫 실행에서 물음이 떠 그대로 멈춘다 |
| `args[1]` | `"@modelcontextprotocol/server-filesystem"` | **띄울 서버 패키지 이름**. npm 에 공개된 이름 그대로다 |
| `args[2]` | `str(DAY_DIR)` | **이 서버가 볼 수 있는 폴더**. 서버가 정한 인자라 서버마다 다르다. 이 밖의 경로는 서버가 거부한다 |
| `transport` | `"stdio"` | 서버와 **어떻게 대화할지**. `stdio` 는 내 컴퓨터에 자식 프로세스로 띄우고 표준입출력으로 주고받는다. 원격 서버라면 `"streamable_http"` 와 `url` 을 쓴다(과제에서 다룬다) |

`args` 의 세 번째 값이 왜 **폴더**인지가 중요합니다. MCP 서버는 저마다 자기 실행 인자를 정합니다.
파일시스템 서버는 "어디까지 열어 줄지"를 인자로 받도록 만들어져 있고, 그게 그대로 **안전장치**가 됩니다.
어떤 인자를 받는지는 그 서버의 공식 문서에서 확인합니다.

In [ ]:
# 파일시스템 서버: 정해 준 폴더의 파일 목록·읽기·쓰기 도구를 내준다.
FILESYSTEM = {
    "command": "npx",                                    # 서버를 띄울 실행기(Node 패키지를 받아 실행한다)
    "args": ["-y",                                       # 설치할지 묻지 않고 진행
             "@modelcontextprotocol/server-filesystem",  # 띄울 서버 패키지 이름
             str(DAY_DIR)],                              # 서버가 볼 수 있는 폴더. 이 밖은 건드리지 못한다
    "transport": "stdio",                                # 내 컴퓨터에 프로세스로 띄우고 표준입출력으로 대화
}

print("허용할 폴더:", FILESYSTEM["args"][2])

---
## 1. 서버에 붙어 도구 목록 받기

연결 설정을 클라이언트에 넘기면 서버가 뜨고, 그 서버가 가진 **도구 목록**이 돌아옵니다.
`{"files": FILESYSTEM}` 의 `"files"` 는 우리가 붙이는 **별명**입니다. 서버가 여럿일 때 구분하려고 씁니다.

`.py` 판은 `async with client.session("files")` 블록 안에서 도구를 받아 썼습니다.
노트북에서는 그 블록이 **셀이 끝날 때 닫혀** 다음 셀에서 도구가 죽습니다.
대신 **`get_tools()`** 를 씁니다. 도구를 부를 때마다 서버에 잠깐 붙었다 떨어지는 방식이라 셀을 넘나들어도 됩니다.

In [ ]:
print("서버를 띄우는 중입니다(첫 실행은 오래 걸립니다)...")
client = MultiServerMCPClient({"files": FILESYSTEM})

# 여기서 서버 프로세스가 실제로 뜬다. await 를 셀에서 바로 쓴다(asyncio.run 은 주피터에서 못 쓴다).
tools = await client.get_tools(server_name="files")

# 받은 도구를 '이름(인자): 설명 첫 줄' 형태로 찍어, 우리가 만들지 않은 도구의 명세를 확인한다.
print(f"도구 {len(tools)}개")
for tool in tools:
    print(f" - {tool.name}({', '.join(tool.args)}): {tool.description.strip().splitlines()[0][:60]}")

---
## 2. 도구의 명세 읽기: 에이전트가 보는 정보

에이전트는 도구를 고를 때 **이름·설명·인자** 세 가지만 봅니다. 19일차에 우리가 `@tool` 로 만든 도구와 구조가 같습니다.
남의 서버가 준 도구든 우리가 만든 도구든, 에이전트 입장에서는 **구분이 없습니다**.

In [ ]:
by_name = {tool.name: tool for tool in tools}     # 이름으로 꺼내 쓰려고 딕셔너리로
listing_tool = by_name["list_directory"]

print("name       :", listing_tool.name)
print("description:", listing_tool.description.strip().splitlines()[0])
print("args       :", listing_tool.args)          # 인자 이름·타입. 도구를 부를 때 이 이름을 그대로 쓴다

---
## 3. 도구를 직접 호출하기: 서버가 준 원래 값

`list_directory` 는 폴더 경로 하나를 받아 그 안의 목록을 돌려줍니다.
**MCP 도구는 비동기 전용**이라 `invoke` 가 아니라 `ainvoke` 로 부르고 `await` 를 붙입니다.
인자 이름은 앞에서 본 `args` 가 알려 준 그대로 씁니다.

In [ ]:
# 돌아온 값을 가공 없이 찍어, MCP 도구의 반환 형태(콘텐츠 블록 리스트)를 눈으로 확인한다.
result = await listing_tool.ainvoke({"path": str(DATA_DIR)})
pprint(result)

> 문자열이 아니라 **`[{'type': 'text', 'text': '...'}]`** 모양입니다. MCP 는 글자 말고 이미지도 돌려줄 수 있어서
> "블록" 이라는 봉투에 담아 보냅니다. 그래서 반환값을 문자열처럼 다루면(`.split()` 등) 그 자리에서 터집니다.

---
## 4. 그 값에서 사람이 읽을 부분 꺼내기

블록 리스트에서 `text` 만 꺼내면 됩니다. `utils.py` 의 `block_text()` 가 그 일을 합니다.
서버가 에러를 낼 때는 **문자열 하나**로 오기도 해서, 세 가지 모양을 모두 받도록 되어 있습니다.

In [ ]:
print(result[0]["text"])          # 블록에서 직접 꺼내기
print("-" * 40)
print(block_text(result))         # 어떤 모양으로 와도 문자열로 만들어 주는 도우미

### 🖐️ 함께 따라하기: 다른 폴더를 다른 도구로 살펴보기

데모는 `data` 폴더를 `list_directory` 로 봤습니다. 이번엔 **`images` 폴더**를 **다른 도구**로 살펴봅니다.

1. `by_name` 에서 **`list_directory_with_sizes`** 도구를 꺼내세요(앞 출력에 이름이 있습니다).
2. `path` 인자에 `DAY_DIR / "images"` 를 문자열로 넘겨 `await` 로 호출하세요.
3. 반환값을 `block_text()` 로 문자열로 만들어 출력하세요.
4. 이어서 **`get_file_info`** 로 `DAY_DIR / "data" / "cvs_sales.csv"` 의 정보를 받아 출력하세요.

**확인 기준**: `images` 폴더의 파일이 **크기와 함께** 나오고, `get_file_info` 출력에 `size`·`modified` 같은 항목이 보입니다.
도구 이름과 인자 이름은 **앞에서 찍어 본 목록** 그대로여야 합니다. 외우지 말고 그 출력을 보고 쓰세요.

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) by_name 에서 list_directory_with_sizes 도구를 꺼낸다
# 2) path 에 DAY_DIR / "images" 를 문자열로 넘겨 await 로 호출한다
# 3) block_text 로 문자열을 뽑아 출력한다
# 4) get_file_info 로 data/cvs_sales.csv 의 정보를 받아 출력한다

---
## 5. 허용된 폴더 밖은 막힌다

서버에 열어 준 폴더는 일차 폴더 하나뿐입니다. 그 밖인 홈 디렉터리를 일부러 불러, 서버가 어떻게 막는지 봅니다.
**거부되는 것이 정상 동작**입니다.

In [ ]:
# 서버는 예외를 던지지 않고 '거부했다'는 문장을 돌려준다. 그래서 반환값을 읽어 확인한다.
denied = await listing_tool.ainvoke({"path": str(Path.home())})
print(block_text(denied))

> 이것이 MCP 서버가 제공하는 **경계**입니다. 우리가 만든 도구라면 이런 검사를 직접 짜야 하지만,
> 서버를 골라 쓰면 그 서버가 정한 안전장치를 그대로 얻습니다. 반대로 말하면 **어떤 서버를 붙일지가 곧 권한 설계**입니다.

> **한글 폴더에서 주의할 점**: 맥은 파일 이름의 한글을 자모가 분리된 형태로 저장합니다.
> 이 경로 문자열이 모델을 거쳐 되돌아오면 글자가 달라져 거부되는 일이 있습니다.
> 에이전트에게 경로를 맡길 때는 **절대경로 대신 `data/cvs_sales.csv` 같은 상대경로**를 쓰게 하면 이 문제를 피할 수 있습니다.

---
# 6. 받은 도구를 LangChain 에이전트에 붙이기

여기까지는 **우리가** 도구를 골라 불렀습니다. 이제 그 일을 모델에게 넘깁니다.
`create_agent(model, tools, system_prompt=...)` 의 `tools` 자리에 **MCP 도구 리스트를 그대로** 넣으면 끝입니다.
19일차에 우리가 `@tool` 로 만든 도구를 넣던 바로 그 자리입니다. 에이전트에게는 둘의 구분이 없습니다.

**도구를 골라서 넘기는 것도 설계입니다.** 14개를 다 넘기면 모델이 설명을 전부 읽어야 해서 비용이 커지고,
`write_file`·`move_file` 처럼 **바꾸는 도구**까지 쥐여 주게 됩니다.
먼저 **읽기 전용**으로 붙여 봅니다.

In [ ]:
# 읽기만 하는 도구 세 개만 골라 넘긴다. 이 에이전트는 파일을 바꿀 수단 자체가 없다.
read_only = [by_name[name] for name in ["list_directory", "read_text_file", "get_file_info"]]
print("넘길 도구:", [t.name for t in read_only])

model = ChatOpenAI(model="gpt-4o-mini", temperature=0)

agent = create_agent(
    model,
    read_only,
    system_prompt=(
        "너는 파일을 살펴보는 조사 담당이다. 파일 관련 질문은 반드시 주어진 파일 도구로 확인하고, "
        "내용을 지어내지 않는다. 경로는 data/cvs_sales.csv 처럼 상대경로로 쓴다. "
        "확인한 내용을 근거로 한국어로 간결히 답한다."
    ),
)
print("에이전트 준비 완료")

시스템 프롬프트에 **"경로는 상대경로로 쓴다"** 를 넣은 이유가 있습니다.
앞에서 본 한글 경로 문제입니다. 절대경로에는 `엔코아_머신러닝` 같은 한글이 들어가는데,
그 문자열이 모델을 거쳐 되돌아오면 글자 모양이 달라져 서버가 **허용 폴더 밖**으로 판정하는 일이 있습니다.
상대경로는 한글 구간을 지나지 않으므로 그 사고가 생기지 않습니다.

도구가 MCP 면 **에이전트도 `ainvoke`** 로 부릅니다.

In [ ]:
question = ("data 폴더에 어떤 파일이 있는지 확인하고, cvs_sales.csv 의 첫 3줄을 읽어서 "
            "이 데이터가 무엇을 담고 있는지 설명해 줘.")
print("질문:", question, "\n")

result = await agent.ainvoke({"messages": question})
print_trajectory(result)

> 기록에서 **모델이 도구를 두 번 이어 부른 것**을 확인하세요. 목록을 먼저 보고, 거기서 확인한 파일 이름으로 읽습니다.
> 우리가 순서를 정해 주지 않았는데도 그렇게 합니다. 앞 도구의 **결과가 다음 도구의 근거**가 되는 것이 에이전트의 핵심 동작입니다.

> `read_text_file` 의 `head` 인자를 모델이 알아서 쓴 것도 눈여겨보세요.
> 우리가 알려 준 적이 없습니다. **도구 명세**, 곧 앞에서 읽어 본 그 `description` 과 `args` 에 적혀 있어서 모델이 읽고 쓴 것입니다.

따라하기에서는 **파일을 새로 만드는 일**과 **이미 있는 파일을 고치는 일**을 시켜 봅니다.
고칠 대상이 있어야 하니, 아래 셀로 **초안 파일**을 하나 만들어 둡니다.
마지막 줄이 비어 있는 시입니다.

In [ ]:
# 에이전트가 고칠 대상 파일을 파이썬으로 미리 만들어 둔다(따라하기 준비물).
DRAFT = DAY_DIR / "output" / "시_초안.md"
DRAFT.parent.mkdir(exist_ok=True)
DRAFT.write_text(
    "# 데이터 분석가의 하루\n\n"
    "아침마다 쌓인 표를 열어\n"
    "빈칸을 세고 이상한 값을 지운다\n"
    "(여기에 마지막 행을 채워 주세요)\n",
    encoding="utf-8")

print(DRAFT.read_text(encoding="utf-8"))

### 🖐️ 함께 따라하기: 시를 쓰고, 이미 있는 파일을 고치기

읽기 전용 에이전트는 파일을 만들지도 고치지도 못합니다. 이번엔 **쓰기 도구와 수정 도구를 얹어** 두 가지 일을 시킵니다.

1. `read_only` 에 `by_name["write_file"]` 과 `by_name["edit_file"]` 을 더한 리스트로 새 에이전트를 만드세요.
   시스템 프롬프트에는 "경로는 상대경로로 쓴다" 를 그대로 넣으세요.
2. 다음 두 가지를 한 번에 시키세요.
   - `output/시_초안.md` 를 읽고, **`(여기에 마지막 행을 채워 주세요)` 줄을** 앞 세 줄과 어울리는 시 한 줄로 **바꾸기**
   - `output/시.md` 에 **데이터를 주제로 한 4행 시**를 새로 써서 저장하기
3. `print_trajectory()` 로 기록을 찍으세요.
4. 파이썬으로 두 파일을 **직접 읽어** 출력하세요.

**확인 기준**: 기록에 `read_text_file`·`edit_file`·`write_file` 이 모두 보이고(순서는 달라질 수 있습니다), 4번에서 초안 파일의
`(여기에 마지막 행을 채워 주세요)` 가 **사라져 있으며** `output/시.md` 가 새로 만들어져 있습니다.
모델의 "저장했습니다" 라는 말만 믿지 말고 **파일로 확인**하는 것이 요령입니다.

**여기서 배우는 것**: 새로 쓰기(`write_file`)와 고치기(`edit_file`)는 다른 도구입니다.
고치기는 **어느 글자를 무엇으로 바꿀지** 정확히 지목해야 해서, 모델이 먼저 파일을 **읽어야** 합니다.
그래서 기록에 읽기가 먼저 나옵니다.

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) read_only 에 write_file 과 edit_file 을 더한 도구 리스트로 새 에이전트를 만든다
# 2) 시_초안.md 의 빈 줄을 시 한 줄로 바꾸고, output/시.md 에 4행 시를 새로 쓰게 시킨다
# 3) print_trajectory 로 기록을 찍는다
# 4) 파이썬으로 두 파일을 직접 읽어 출력한다

---
## 이번 실습 정리

| 배운 것 | 요점 |
|---|---|
| 서버 설정 | `command`·`args`·`transport` 딕셔너리 하나가 "서버를 어떻게 띄울지" 전부다 |
| 도구 받기 | `MultiServerMCPClient` 에 설정을 주고 `get_tools()`. 노트북에서는 `session()` 블록보다 이쪽이 편하다 |
| 도구 명세 | 이름·설명·인자 세 가지. 에이전트가 도구를 고를 때 보는 정보와 같다 |
| 호출과 반환 | `ainvoke` + `await`, 반환은 콘텐츠 블록 리스트. `block_text()` 로 문자열을 뽑는다 |
| 안전장치 | 서버가 정한 경계(허용 폴더)가 그대로 우리 안전장치가 된다 |
| 에이전트에 붙이기 | `create_agent(model, tools, ...)` 의 `tools` 자리에 MCP 도구를 그대로 넣는다. **골라 넘기는 것이 곧 권한 설계** |

다음 실습: `02_웹검색_MCP.ipynb` 에서 검색 서버를 **에이전트에 붙여**, 무엇을 부를지 모델이 정하게 합니다.